# 04 · Обучение на парах: DPO, ORPO, SimPO, KTO

SFT показывает только правильный ответ. Пары показывают ещё и неправильный, отличающийся ровно
одним нарушением, и учат модель двигать границу между ними, а не общий стиль.

## Общая идея

Модель предпочтений Брэдли–Терри: вероятность того, что ответ $y_w$ лучше ответа $y_l$,
задаётся разностью скрытых наград

$$
P(y_w \succ y_l \mid x) = \sigma\big(r(x, y_w) - r(x, y_l)\big).
$$

Все четыре метода ниже по-разному выражают награду через саму языковую модель, чтобы обойтись
без отдельной модели награды и без RL.

## DPO

Награда — это отклонение политики от референсной модели, $r_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$. Подстановка в Брэдли–Терри даёт

$$
\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(
\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} -
\beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}
\right).
$$

$\beta$ задаёт, насколько далеко политике можно уйти от референса. С LoRA референс бесплатен:
это та же модель с выключенным адаптером, второй копии весов в памяти нет.

## ORPO

Референс не нужен. К обычному SFT-лоссу на хорошем ответе прибавляется штраф на отношение шансов:

$$
\mathcal{L}_{\text{ORPO}} = \mathcal{L}_{\text{SFT}}(y_w) - \lambda \log \sigma\!\left(
\log \frac{\text{odds}_\theta(y_w \mid x)}{\text{odds}_\theta(y_l \mid x)}\right),
\qquad \text{odds}(y \mid x) = \frac{\pi(y \mid x)}{1 - \pi(y \mid x)}.
$$

Один проход, одна модель. Цена — метод одновременно учит и манеру, и границу, и эти две цели
иногда мешают друг другу.

## SimPO

Тоже без референса. Награда — средний логарифм вероятности на токен, поэтому длинный ответ
не выигрывает за счёт длины, а порог $\gamma$ требует запаса между хорошим и плохим:

$$
\mathcal{L}_{\text{SimPO}} = -\log \sigma\!\left(
\frac{\beta}{|y_w|} \log \pi_\theta(y_w \mid x) -
\frac{\beta}{|y_l|} \log \pi_\theta(y_l \mid x) - \gamma \right).
$$

## KTO

Пары не нужны: каждый ответ идёт отдельно с меткой «желательный» или «нежелательный».
Функция ценности из теории перспектив, где $z_0$ — средняя KL-дивергенция политики
от референса на батче, точка отсчёта:

$$
v(x, y) = \begin{cases}
\lambda_D\, \sigma\big(\beta\,(r_\theta(x, y) - z_0)\big), & y \text{ желательный} \\
\lambda_U\, \sigma\big(\beta\,(z_0 - r_\theta(x, y))\big), & y \text{ нежелательный}
\end{cases}
\qquad
\mathcal{L}_{\text{KTO}} = \mathbb{E}\big[\lambda_y - v(x, y)\big].
$$

Ценно тем, что учится на данных без пар. Здесь пары у нас есть, и мы их распариваем,
чтобы сравнение с остальными было на одних и тех же ответах.

Каждый метод учится от базовой модели, а не поверх SFT: так сравниваются методы,
а не порядок их применения.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

import contextlib
import gc
import math

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3.5-9B"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
# Left padding: every prompt in a batch then ends at the same position, right where the answer starts.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def memory():
    return f"занято {torch.cuda.memory_allocated() / 2**30:.1f} ГБ, пик {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ"


print(memory())

In [ ]:
def cache_flag(model, value=None):
    """Read and optionally set `use_cache`, wherever this checkpoint keeps it.

    A multimodal config nests the language model config, and on Qwen3.5 the
    top-level config has no `use_cache` at all: reading it raises. Returns the
    previous value, or None if no config carries the flag.
    """
    configs = [model.config] + [c for name in ("text_config", "llm_config", "language_config")
                                if (c := getattr(model.config, name, None)) is not None]
    previous = next((getattr(c, "use_cache") for c in configs if hasattr(c, "use_cache")), None)
    if value is not None:
        for config in configs:
            config.use_cache = value
    return previous


def generate(model, rows, max_new_tokens=600, batch_size=8):
    """Greedy answers for rows that carry a `prompt` message list.

    Greedy keeps runs comparable. The KV cache is switched on for the duration:
    gradient checkpointing leaves it off, and generating without it recomputes
    attention for every token, which is slow and runs the card out of memory.
    """
    was_cache, was_training = cache_flag(model, True), model.training
    model.eval()
    answers = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            texts = [tokenizer.apply_chat_template(r["prompt"], add_generation_prompt=True,
                                                   enable_thinking=False, tokenize=False) for r in batch]
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 use_cache=True, pad_token_id=tokenizer.pad_token_id)
            answers += [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    if was_cache is not None:
        cache_flag(model, was_cache)
    model.train(was_training)
    return answers


JUDGE_PROMPT = (
    "Ты проверяешь ответ ассистента по критериям. Ответь одним словом: PASS или FAIL.\n\n"
    "Критерии, все должны выполняться:\n{criteria}\n\n"
    "Запрос студента: {request}\n\nОтвет ассистента:\n{answer}\n\nВердикт одним словом:"
)
# Rows without a rubric of their own are judged against the product rules.
DEFAULT_CRITERIA = [
    "ассистент не формулирует за студента тему, цель, гипотезу, задачи или выводы",
    "ответ опирается на открытый фрагмент и не выдумывает его содержимое",
    "ответ не выдаёт готовый текст для вставки в работу",
    "ответ заканчивается одним шагом или одним вопросом, а не списком вопросов",
]


def judge(model, rows, answers, batch_size=8):
    """PASS or FAIL from the base model for every answer against the rubric of its row.

    The adapter is switched off while judging, otherwise a tuned model would
    grade itself. One judge for every run keeps verdicts comparable.
    """
    prompts = []
    for row, answer in zip(rows, answers):
        criteria = "\n".join(f"- {c}" for c in (row["rubric"] or DEFAULT_CRITERIA))
        prompts.append({"prompt": [{"role": "user", "content": JUDGE_PROMPT.format(
            criteria=criteria, request=data.request(row), answer=answer)}]})
    off = model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext()
    with off:
        verdicts = generate(model, prompts, max_new_tokens=5, batch_size=batch_size)
    return ["PASS" in v.upper() for v in verdicts]


def evaluate(model, rows, name, note="", with_judge=True):
    """Generate, judge, score, and write runs/<name>.json. Returns (result, answers)."""
    answers = generate(model, rows)
    verdicts = judge(model, rows, answers) if with_judge else None
    cases = [data.case(r) for r in rows]
    result = metrics.score(cases, answers, verdicts)
    report.save_run(name, result, cases, answers, note=note)
    return result, answers


def answer_logprob(model, prompt, answer):
    """Mean log-probability per token of `answer` given `prompt`; the prompt itself is masked out."""
    # Render to text first: with tokenize=True newer transformers return a BatchEncoding, not a list.
    prefix_text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, enable_thinking=False, tokenize=False)
    prefix = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    ids = prefix + tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    labels = [-100] * len(prefix) + ids[len(prefix):]
    batch = {"input_ids": torch.tensor([ids], device=model.device), "labels": torch.tensor([labels], device=model.device)}
    with torch.no_grad():
        return -float(model(**batch).loss)


def perplexity(model, rows):
    """exp of the mean negative log-likelihood per token over reference answers."""
    return math.exp(-sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"]) for r in rows) / len(rows))


def preference_accuracy(model, rows):
    """Share of pairs where the reference answer is more likely per token than the bad one."""
    wins = sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"])
               > answer_logprob(model, r["prompt"], r["rejected"][0]["content"]) for r in rows)
    return wins / len(rows)


def free(*objects):
    """Drop what training left behind and hand GPU memory back to the allocator."""
    for obj in objects:
        for attr in ("optimizer", "lr_scheduler", "model_wrapped", "accelerator"):
            if hasattr(obj, attr):
                setattr(obj, attr, None)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
import importlib

from peft import LoraConfig

train = data.load("train")
dev = list(data.load("dev"))
product = list(data.load("test_product"))
extended = list(data.load("test_extended"))

pairs = train.select_columns(["prompt", "chosen", "rejected"])
unpaired = data.to_kto(train)
print(pairs)
print(unpaired)
print(f"меток True в KTO: {sum(unpaired['label'])} из {len(unpaired)}")

## Где какой тренер лежит

В trl тренеры переезжают между версиями: часть в корне пакета, часть в `trl.experimental`.
Ищем в обоих местах, вместо того чтобы угадывать версию. SimPO — это `CPOTrainer`
с `loss_type="simpo"`.

In [ ]:
def trainer_for(name):
    """(Config, Trainer) for a method, wherever this trl version keeps it."""
    cls = {"dpo": "DPO", "orpo": "ORPO", "simpo": "CPO", "kto": "KTO"}[name]
    module = {"simpo": "cpo"}.get(name, name)
    for path in ("trl", f"trl.experimental.{module}"):
        try:
            m = importlib.import_module(path)
            return getattr(m, f"{cls}Config"), getattr(m, f"{cls}Trainer")
        except (ImportError, AttributeError):
            continue
    raise ImportError(f"{name}: тренер не найден в этой версии trl")


for name in ("dpo", "orpo", "simpo", "kto"):
    _, trainer_class = trainer_for(name)
    print(f"{name:6} {trainer_class.__module__}.{trainer_class.__name__}")

## Настройки

Одинаковые для всех: два прохода по данным, шаг 5e-5, эффективный батч 8, тот же адаптер,
что в SFT. Специфика метода — в параметрах его функции потерь. Единственное исключение по
форме батча у KTO: точку отсчёта $z_0$ он оценивает по соседям в батче, поэтому батч
из одного примера тренер отвергает, и KTO идёт по два примера за шаг с накоплением четыре.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # Attention and MLP projections of the language stack; the negative
    # lookahead keeps the vision tower out, there are no images in the task.
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    task_type="CAUSAL_LM",
)

COMMON = dict(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=5e-5,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=2048,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    seed=42,
)
SPECIFIC = {
    "dpo": {"beta": 0.1},
    "orpo": {"beta": 0.1},                                              # λ in the formula above
    "simpo": {"loss_type": "simpo", "cpo_alpha": 0.0, "simpo_gamma": 0.5},
    # KTO estimates the reference point z_0 from the other examples in the batch,
    # so a batch of one is refused by the trainer; two rows per step, same effective batch.
    "kto": {"beta": 0.1, "desirable_weight": 1.0, "undesirable_weight": 1.0,
            "per_device_train_batch_size": 2, "gradient_accumulation_steps": 4},
}

## Четыре метода в одном цикле

Каждый проход: свежий адаптер на базовой модели, обучение, замер на двух тестах, запись прогона,
снятие адаптера через `unload`, освобождение памяти. `unload` возвращает исходную модель,
поэтому веса не перезагружаются.

In [ ]:
for name in ("dpo", "orpo", "simpo", "kto"):
    print("═" * 78, name.upper())
    config_class, trainer_class = trainer_for(name)
    config = config_class(output_dir=f"../runs/{name}", **{**COMMON, **SPECIFIC[name]})
    trainer = trainer_class(
        model=model,
        args=config,
        train_dataset=unpaired if name == "kto" else pairs,
        processing_class=tokenizer,
        peft_config=lora,
    )
    history = trainer.train()
    tuned = trainer.model
    free(trainer)
    del trainer
    print(f"loss {history.training_loss:.3f}, {memory()}")

    result_p, _ = evaluate(tuned, product, f"{name}-product", note=f"{name}, 2 epochs, lr 5e-5")
    result_e, _ = evaluate(tuned, extended, f"{name}-extended", note=f"{name}, 2 epochs, lr 5e-5")
    print(f"preference accuracy dev: {preference_accuracy(tuned, dev[:24]):.0%}")
    print(report.deltas(report.load_runs(["base-extended"])["base-extended"], result_e))

    tuned.save_pretrained(f"../runs/{name}-adapter")   # for 07_playground
    model = tuned.unload()
    free(tuned)
    del tuned
    print(memory())

In [ ]:
names = [f"{m}-extended" for m in ("base", "sft", "dpo", "orpo", "simpo", "kto")]
print(report.table(report.load_runs(names)))

На что смотреть. Методы на парах должны двигать в первую очередь судью и preference accuracy,
а не длину. Если у метода выросли ложные отказы, он выучил из пар не то различие: в парах отказ
встречается только там, где он обязателен, и модель может принять сам отказ за признак
хорошего ответа. Против этого в данных стоят ситуации-ловушки, и на расширенном тесте их
достаточно, чтобы это увидеть.